In [ ]:
# ==================== IMPROVEMENTS TO ADD TO YOUR EXISTING MODEL ====================
# Add these functions to your current code for better performance

import pandas as pd
import numpy as np
import re

# ==================== IMPROVEMENT 1: Better Unit Extraction ====================
def extract_enhanced_unit_info(text):
    """More comprehensive unit extraction with better patterns"""
    if pd.isna(text):
        return {'unit_value': 0, 'unit_type': 'none', 'oz_equiv': 0, 'is_liquid': 0}
    
    text_str = str(text).lower()
    
    # Liquid units
    liquid_patterns = [
        (r'(\d+\.?\d*)\s*fl\.?\s*oz', 'fl_oz', 1, 1),
        (r'(\d+\.?\d*)\s*fluid ounce', 'fl_oz', 1, 1),
        (r'(\d+\.?\d*)\s*ml\b', 'ml', 0.033814, 1),
        (r'(\d+\.?\d*)\s*milliliter', 'ml', 0.033814, 1),
        (r'(\d+\.?\d*)\s*liter', 'liter', 33.814, 1),
        (r'(\d+\.?\d*)\s*litre', 'liter', 33.814, 1),
        (r'(\d+\.?\d*)\s*gallon', 'gallon', 128, 1),
    ]
    
    # Solid units
    solid_patterns = [
        (r'(\d+\.?\d*)\s*ounce(?!s)', 'ounce', 1, 0),
        (r'(\d+\.?\d*)\s*oz\b', 'ounce', 1, 0),
        (r'(\d+\.?\d*)\s*pound', 'pound', 16, 0),
        (r'(\d+\.?\d*)\s*lb\b', 'pound', 16, 0),
        (r'(\d+\.?\d*)\s*gram', 'gram', 0.035274, 0),
        (r'(\d+\.?\d*)\s*kg\b', 'kg', 35.274, 0),
    ]
    
    # Try liquid patterns first
    for pattern, unit, conversion, is_liquid in liquid_patterns:
        match = re.search(pattern, text_str)
        if match:
            value = float(match.group(1))
            return {
                'unit_value': value,
                'unit_type': unit,
                'oz_equiv': value * conversion,
                'is_liquid': is_liquid
            }
    
    # Then solid patterns
    for pattern, unit, conversion, is_liquid in solid_patterns:
        match = re.search(pattern, text_str)
        if match:
            value = float(match.group(1))
            return {
                'unit_value': value,
                'unit_type': unit,
                'oz_equiv': value * conversion,
                'is_liquid': is_liquid
            }
    
    return {'unit_value': 0, 'unit_type': 'none', 'oz_equiv': 0, 'is_liquid': 0}

# ==================== IMPROVEMENT 2: Price Range Features ====================
def add_price_range_features(train_df, test_df):
    """Add features based on typical price ranges for different product categories"""
    
    def get_price_indicators(text):
        if pd.isna(text):
            return {'bulk_indicator': 0, 'premium_indicator': 0, 'single_serve': 0}
        
        text_lower = str(text).lower()
        
        # Bulk indicators (usually cheaper per unit)
        bulk_words = ['bulk', 'wholesale', 'case of', 'pack of 12', 'pack of 24', '12-pack', '24-pack']
        bulk_score = sum(1 for word in bulk_words if word in text_lower)
        
        # Premium indicators (usually more expensive)
        premium_words = ['organic', 'gourmet', 'artisan', 'premium', 'imported', 'specialty', 
                        'handcrafted', 'natural', 'grass-fed', 'free-range', 'non-gmo']
        premium_score = sum(1 for word in premium_words if word in text_lower)
        
        # Single serve (usually cheaper)
        single_words = ['single', 'individual', 'travel size', 'sample', 'trial']
        single_score = sum(1 for word in single_words if word in text_lower)
        
        return {
            'bulk_indicator': min(bulk_score, 3),
            'premium_indicator': min(premium_score, 3),
            'single_serve': min(single_score, 2)
        }
    
    train_indicators = train_df['catalog_content'].apply(get_price_indicators)
    test_indicators = test_df['catalog_content'].apply(get_price_indicators)
    
    for key in ['bulk_indicator', 'premium_indicator', 'single_serve']:
        train_df[key] = train_indicators.apply(lambda x: x[key])
        test_df[key] = test_indicators.apply(lambda x: x[key])
    
    return train_df, test_df

# ==================== IMPROVEMENT 3: Better Pack Size Extraction ====================
def extract_comprehensive_pack_info(text):
    """Extract pack size with multiple fallback patterns"""
    if pd.isna(text):
        return {'pack_qty': 1, 'pack_confidence': 0}
    
    text_str = str(text).lower()
    
    # High confidence patterns
    high_conf_patterns = [
        r'\(pack of (\d+)\)',
        r'pack of (\d+)',
        r'\((\d+) pack\)',
        r'(\d+)-pack',
        r'(\d+) pack\b',
    ]
    
    for pattern in high_conf_patterns:
        match = re.search(pattern, text_str)
        if match:
            qty = float(match.group(1))
            if qty > 1:
                return {'pack_qty': qty, 'pack_confidence': 2}
    
    # Medium confidence patterns
    value_match = re.search(r'value:\s*(\d+\.?\d*)', text_str)
    if value_match:
        val = float(value_match.group(1))
        if val > 1:
            return {'pack_qty': val, 'pack_confidence': 1}
    
    # Low confidence patterns
    count_match = re.search(r'(\d+)\s*count', text_str)
    if count_match:
        count = float(count_match.group(1))
        if count > 1 and count <= 100:
            return {'pack_qty': count, 'pack_confidence': 0.5}
    
    return {'pack_qty': 1, 'pack_confidence': 0}

# ==================== IMPROVEMENT 4: Brand Name Extraction ====================
def extract_brand_name(text):
    """Extract potential brand name from item title"""
    if pd.isna(text):
        return 'UNKNOWN'
    
    # Extract item name
    item_match = re.search(r'Item Name:\s*([^\n]+)', str(text))
    if not item_match:
        return 'UNKNOWN'
    
    item_name = item_match.group(1).strip()
    
    # Brand is usually the first 1-2 capitalized words
    words = item_name.split()
    if not words:
        return 'UNKNOWN'
    
    # Take first capitalized word(s)
    brand_words = []
    for word in words[:3]:  # Check first 3 words max
        if word and word[0].isupper():
            brand_words.append(word)
        else:
            break
    
    if brand_words:
        brand = ' '.join(brand_words[:2])  # Max 2 words for brand
        # Clean common non-brand words
        non_brands = ['The', 'A', 'An', 'Pack', 'Set', 'Box', 'Item']
        if brand not in non_brands:
            return brand
    
    return 'UNKNOWN'

# ==================== IMPROVEMENT 5: Advanced Interaction Features ====================
def create_interaction_features(df):
    """Create meaningful interaction features"""
    
    # Pack × Unit interactions
    df['total_volume'] = df['pack_quantity'] * df['unit_oz_equiv']
    df['log_total_volume'] = np.log1p(df['total_volume'])
    
    # Price-relevant ratios
    df['volume_per_dollar_proxy'] = df['log_total_volume'] / (df['text_length'] + 1) * 1000
    
    # Premium × Volume (premium products in bulk are expensive)
    df['premium_volume'] = df['is_premium'] * df['log_total_volume']
    
    # Bulk × Unit size (bulk of small items vs large items)
    df['bulk_size_interaction'] = df['is_multi_pack'] * df['unit_oz_equiv']
    
    # Brand reputation proxy (long brand name = established brand)
    if 'brand_name' in df.columns:
        df['brand_name_length'] = df['brand_name'].str.len()
    
    # Text richness (detailed descriptions often mean premium products)
    df['text_richness'] = df['word_count'] / (df['bullet_count'] + 1)
    
    return df

# ==================== IMPROVEMENT 6: Enhanced TF-IDF ====================
from sklearn.feature_extraction.text import TfidfVectorizer

def create_enhanced_tfidf(train_texts, test_texts):
    """Enhanced TF-IDF with better parameters"""
    
    # Extract just the item names for focused TF-IDF
    train_names = train_texts.str.extract(r'Item Name:([^\n]+)', expand=False).fillna('')
    test_names = test_texts.str.extract(r'Item Name:([^\n]+)', expand=False).fillna('')
    
    # Extract bullet points
    def get_bullets(text):
        if pd.isna(text):
            return ''
        bullets = re.findall(r'Bullet Point[^:]*:\s*([^\n]+)', str(text))
        return ' '.join(bullets)
    
    train_bullets = train_texts.apply(get_bullets)
    test_bullets = test_texts.apply(get_bullets)
    
    # Item name TF-IDF (most important for price)
    tfidf_name = TfidfVectorizer(
        max_features=2000,  # Increased from 1000
        ngram_range=(1, 4),  # Added 4-grams
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        stop_words='english',
        token_pattern=r'\b[a-zA-Z][a-zA-Z]+\b'  # Better tokenization
    )
    
    # Bullet points TF-IDF
    tfidf_bullets = TfidfVectorizer(
        max_features=2000,  # Increased from 1000
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        stop_words='english'
    )
    
    # Character-level for brand names
    tfidf_char = TfidfVectorizer(
        max_features=1000,  # Increased from 500
        analyzer='char',
        ngram_range=(3, 6),  # Extended range
        min_df=5,
        sublinear_tf=True
    )
    
    train_tfidf_name = tfidf_name.fit_transform(train_names).toarray()
    test_tfidf_name = tfidf_name.transform(test_names).toarray()
    
    train_tfidf_bullets = tfidf_bullets.fit_transform(train_bullets).toarray()
    test_tfidf_bullets = tfidf_bullets.transform(test_bullets).toarray()
    
    train_tfidf_char = tfidf_char.fit_transform(train_names).toarray()
    test_tfidf_char = tfidf_char.transform(test_names).toarray()
    
    train_combined = np.hstack([train_tfidf_name, train_tfidf_bullets, train_tfidf_char])
    test_combined = np.hstack([test_tfidf_name, test_tfidf_bullets, test_tfidf_char])
    
    print(f"✅ Enhanced TF-IDF: {train_combined.shape[1]} features (name: {train_tfidf_name.shape[1]}, bullets: {train_tfidf_bullets.shape[1]}, char: {train_tfidf_char.shape[1]})")
    
    return train_combined, test_combined

# ==================== IMPROVEMENT 7: Better Model Parameters ====================
# Replace your existing model params with these tuned ones

IMPROVED_LGBM_PARAMS = {
    'objective': 'regression',
    'metric': 'mae',
    'n_estimators': 5000,  # Increased from 3000
    'learning_rate': 0.008,  # Slightly lower
    'max_depth': 12,  # Deeper trees
    'num_leaves': 100,  # More leaves
    'min_child_samples': 15,  # Less restrictive
    'subsample': 0.85,  # Slightly higher
    'subsample_freq': 1,
    'colsample_bytree': 0.85,  # Slightly higher
    'reg_alpha': 0.3,  # Less L1 regularization
    'reg_lambda': 1.5,  # Less L2 regularization
    'min_gain_to_split': 0.01,  # Added
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

IMPROVED_XGB_PARAMS = {
    'objective': 'reg:squarederror',
    'n_estimators': 5000,  # Increased
    'learning_rate': 0.008,  # Lower
    'max_depth': 10,  # Deeper
    'min_child_weight': 2,  # Less restrictive
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.3,
    'reg_lambda': 1.5,
    'gamma': 0.05,  # Less restrictive
    'random_state': 43,
    'tree_method': 'hist',
    'n_jobs': -1,
    'early_stopping_rounds': 300  # Increased patience
}

IMPROVED_CATBOOST_PARAMS = {
    'iterations': 5000,  # Increased
    'learning_rate': 0.008,  # Lower
    'depth': 10,  # Deeper
    'l2_leaf_reg': 2.0,  # Less regularization
    'random_seed': 44,
    'loss_function': 'MAE',
    'verbose': False,
    'early_stopping_rounds': 300
}

# ==================== IMPROVEMENT 8: Weighted Ensemble ====================
def optimized_ensemble_weights():
    """
    Based on your meta-model weights:
    LGBM: 0.4148, XGB: 0.9099, CatBoost: -0.2743
    
    Normalize and adjust for better stability
    """
    # Your meta-model showed XGB is strongest, CatBoost is negatively correlated (diversity!)
    # This is actually good - it means CatBoost catches different patterns
    
    return {
        'lgbm': 0.35,  # Solid contributor
        'xgb': 0.50,   # Strongest model
        'catboost': 0.15  # Keeps some diversity
    }

# ==================== HOW TO USE ====================
"""
INTEGRATION STEPS:

1. Replace your create_advanced_features() function to include:
   - extract_enhanced_unit_info()
   - extract_comprehensive_pack_info()
   - extract_brand_name()
   - Then call create_interaction_features()

2. Add price range features:
   train_df, test_df = add_price_range_features(train_df, test_df)

3. Replace your TF-IDF creation with create_enhanced_tfidf()

4. Update model parameters with IMPROVED_* params

5. In your meta-model, try weighted average using optimized_ensemble_weights()
   before Ridge regression as an additional ensemble member

EXPECTED IMPROVEMENT: 46.2% → 44-45% SMAPE
"""

print("✅ Improvements ready to integrate!")
print("Expected gain: 1-2% SMAPE improvement")
print("Focus areas: Better features + More TF-IDF + Tuned hyperparameters")